# 02 – Opschonen en aggregeren (dinsdagochtend)
Rubric **Opschonen**: elke stap wordt gelogd met *hoeveel* rijen eruit gaan en *waarom*.
Dat logboek (`cleaning_log.csv`) laat je straks zien in het dashboard.

Aan het eind slaan we kleine, geaggregeerde bestanden op die het Streamlit-dashboard snel kan inlezen.

In [ ]:
# Load the settings and data prepared in notebook 01
import os, sys
from pathlib import Path
# The notebooks live in notebooks/: work from the project root so data/ and weather_impact.py are found
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROC = Path("data/processed")
cfg = json.loads((PROC / "config.json").read_text())
START, END = pd.Timestamp(cfg["start"]), pd.Timestamp(cfg["end"])
rides = pd.read_parquet(PROC / "rides_raw.parquet")
stations = pd.read_parquet(PROC / "stations.parquet")
print(cfg)
print(f"{len(rides):,} ritten ingeladen")

## 1. Ritten opschonen

In [ ]:
# Remove invalid rides step by step and log how many rows each step removes and why
log = []
def step(df, keep_mask, reason):
    before = len(df)
    df = df[keep_mask]
    log.append({"step": reason, "before": before, "removed": before - len(df), "after": len(df),
                "pct_removed": round((before - len(df)) / before * 100, 3)})
    print(f"{reason:<45} -{before - len(df):>9,}  → {len(df):,}")
    return df

rides = step(rides, ~rides.duplicated(subset=["ride_id", "start_time"]), "Dubbele ritten (weekbestanden overlappen)")
# Some weekly files have no end station ID, only the end station name (e.g. 06–12 Jul 2022).
# Those rides are kept: the station is linked by name in step 2.
has_end_station = rides["end_station_id"].notna() | rides["end_station"].notna()
rides = step(rides, rides[["start_time", "end_time", "start_station_id"]].notna().all(axis=1) & has_end_station,
             "Ontbrekende tijd of station")
print("Ritten zonder eind-station-ID (koppeling via naam):", int(rides["end_station_id"].isna().sum()))
rides = step(rides, rides["start_time"].between(START, END + pd.Timedelta(days=1), inclusive="left"),
             "Buiten gekozen periode")
rides = step(rides, rides["end_time"] >= rides["start_time"], "Eindtijd vóór starttijd")
rides = step(rides, rides["duration_min"] >= 1, "Rit korter dan 1 minuut (fiets direct terug)")
rides = step(rides, rides["duration_min"] <= 24 * 60, "Rit langer dan 24 uur (verloren/defect)")

cleaning_log = pd.DataFrame(log)
cleaning_log.to_csv(PROC / "cleaning_log.csv", index=False)
cleaning_log

## 2. Stations koppelen aan coördinaten
Oud formaat → koppelen op `bikepoint_id`; nieuw formaat → op `terminal_id`.
Wat dan nog niet matcht, proberen we op stationsnaam.

In [ ]:
# Link every ride to one station ID (bikepoint_id): via terminal number, BikePoint ID or, as a fallback, station name
s = stations.dropna(subset=["terminal_id"])
term2bp = pd.Series(s["bikepoint_id"].values, index=s["terminal_id"].astype(int))
term2bp = term2bp[~term2bp.index.duplicated()]
name2bp = stations.set_index(stations["name"].str.lower().str.strip())["bikepoint_id"]
name2bp = name2bp[~name2bp.index.duplicated()]

def link_station(id_col, name_col):
    ids = rides[id_col].astype("float")
    bp = np.where(rides["id_type"] == "terminal", ids.map(term2bp), ids)
    bp = pd.Series(bp, index=rides.index).where(lambda x: x.isin(stations["bikepoint_id"]))
    via_name = rides[name_col].astype(str).str.lower().str.strip().map(name2bp)
    return bp.fillna(via_name).astype("Int64")

rides["start_bp"] = link_station("start_station_id", "start_station")
rides["end_bp"] = link_station("end_station_id", "end_station")
print(f"Startstation gekoppeld: {rides['start_bp'].notna().mean():.1%}")
print(f"Eindstation gekoppeld:  {rides['end_bp'].notna().mean():.1%}")
print("\nMeest voorkomende niet-gekoppelde stations (waarschijnlijk opgeheven):")
print(rides.loc[rides["start_bp"].isna(), "start_station"].value_counts()[lambda x: x > 0].head(10))

> Niet-gekoppelde ritten blijven in de dag-totalen (voor de regen-analyse), maar vallen weg op de kaart.
> Benoem dit percentage in je presentatie.

## 3. Weerdata opschonen

In [ ]:
# Clean the weather data: rename columns, filter the period, handle missing/impossible values, drop empty columns
w = pd.read_csv(cfg["weather_file"])
dc = w[cfg["date_col"]]
if pd.api.types.is_numeric_dtype(dc):
    w["date"] = pd.to_datetime(dc.astype("Int64").astype(str), format="%Y%m%d", errors="coerce")
else:
    dayfirst = bool(dc.astype(str).str.match(r"^\d{1,2}[/-]").iloc[0])
    w["date"] = pd.to_datetime(dc, errors="coerce", dayfirst=dayfirst)
w["date"] = w["date"].dt.normalize()
w = w.rename(columns={cfg["rain_col"]: "precip_mm", cfg["temp_col"]: "temp_mean"})
if cfg["date_col"] != "date":
    w = w.drop(columns=[cfg["date_col"]])
w = w[w["date"].between(START, END)].drop_duplicates("date")

n_rain_missing = int(w["precip_mm"].isna().sum())
n_negative = int((w["precip_mm"] < 0).sum())
w.loc[w["precip_mm"] < 0, "precip_mm"] = np.nan             # negative precipitation is impossible
w = w.set_index("date").asfreq("D")                          # make missing days visible
n_temp_missing = int(w["temp_mean"].isna().sum())
w["temp_mean"] = w["temp_mean"].interpolate(limit=3)         # fill short gaps in temperature
empty_cols = [c for c in w.columns if w[c].isna().mean() > 0.5]
w = w.drop(columns=empty_cols)                               # drop (almost) empty columns

weather_log = pd.DataFrame([
    {"step": "Neerslag ontbrekend (blijft NaN, niet gegokt)", "count": n_rain_missing},
    {"step": "Negatieve neerslag → NaN", "count": n_negative},
    {"step": "Temperatuur ontbrekend → lineair geïnterpoleerd (max 3 dagen)", "count": n_temp_missing},
    {"step": f"Kolommen >50% leeg verwijderd: {', '.join(empty_cols) or '-'}", "count": len(empty_cols)},
])
weather = w.reset_index()
print(weather_log.to_string(index=False))
weather.describe().T

## 4. Extra kenmerken (features)
- **Regencategorie.** Onder 0,2 mm telt als droog (meetruis/motregen).
- **Weekdag/weekend en Britse feestdagen.** Die zijn nodig om het effect van regen eerlijk te meten.

*De labels (waarden) houden we Nederlands, omdat ze zo in de grafieken van de presentatie komen.*

In [ ]:
# Add features: rain category, weekday/weekend, UK bank holidays, day type and month
try:
    import holidays
    uk_holidays = set(holidays.UK(subdiv="ENG", years=range(START.year, END.year + 1)).keys())
except ImportError:
    print("Tip: pip install holidays  (nu zonder feestdagen)")
    uk_holidays = set()

weather["rain_cat"] = pd.cut(weather["precip_mm"], [-0.01, 0.2, 2, 5, np.inf],
                             labels=["droog (<0,2 mm)", "licht (0,2–2)", "matig (2–5)", "zwaar (>5)"], right=False)
weather["weekday"] = weather["date"].dt.dayofweek          # 0 = Monday
weather["is_weekend"] = weather["weekday"] >= 5
weather["is_holiday"] = weather["date"].dt.date.isin(uk_holidays)
weather["day_type"] = np.where(weather["is_weekend"] | weather["is_holiday"], "weekend/feestdag", "werkdag")
weather["month"] = weather["date"].dt.month
weather["rain_cat"].value_counts().sort_index()

## 5. Aggregeren en opslaan voor het dashboard

In [ ]:
# Aggregate ~10M rides into small tables for the analysis and the Streamlit dashboard
rides["date"] = rides["start_time"].dt.normalize()
rides["hour"] = rides["start_time"].dt.hour

# a) per day + weather
daily = (rides.groupby("date").agg(rides=("ride_id", "size"),
                                   duration_mean=("duration_min", "mean"),
                                   duration_median=("duration_min", "median"))
         .reset_index().merge(weather, on="date", how="left"))
daily.to_parquet(PROC / "daily.parquet", index=False)

# b) per day per hour
hourly = rides.groupby(["date", "hour"]).size().rename("rides").reset_index().merge(
    weather[["date", "rain_cat", "day_type", "precip_mm"]], on="date", how="left")
hourly.to_parquet(PROC / "hourly.parquet", index=False)

# c) per day per start station (for map + sub-question)
daily_station = (rides.dropna(subset=["start_bp"]).groupby(["date", "start_bp"]).size()
                 .rename("departures").reset_index())
daily_station.to_parquet(PROC / "daily_station.parquet", index=False)

# d) totals per station: departures, arrivals, balance
departures = rides.groupby("start_bp").size().rename("departures")
arrivals = rides.groupby("end_bp").size().rename("arrivals")
station_totals = (stations.set_index("bikepoint_id").join(departures).join(arrivals)
                  .fillna({"departures": 0, "arrivals": 0}).reset_index())
station_totals["balance"] = station_totals["arrivals"] - station_totals["departures"]
station_totals["departures_per_day"] = station_totals["departures"] / daily["date"].nunique()
station_totals.to_parquet(PROC / "station_totals.parquet", index=False)

# e) top 500 routes (busiest routes/places)
routes = (rides.dropna(subset=["start_bp", "end_bp"]).groupby(["start_bp", "end_bp"]).size()
          .rename("rides").sort_values(ascending=False).head(500).reset_index())
routes.to_parquet(PROC / "routes_top500.parquet", index=False)

weather_log.to_csv(PROC / "weather_log.csv", index=False)
for f in sorted(PROC.glob("*")):
    print(f"{f.name:<28} {f.stat().st_size / 1e6:8.2f} MB")

In [ ]:
# Preview of the daily table
daily.head()

### Check
- Zijn er dagen zonder ritten of zonder weer? (`daily.isna().sum()`)
- Zijn de bestanden klein genoeg voor GitHub (<100 MB)? `rides_raw.parquet` hoeft **niet** naar GitHub.

In [ ]:
# Final check: remaining missing values and number of days
print(daily.isna().sum()[lambda x: x > 0])
print("Dagen in periode:", (END - START).days + 1, "| dagen met data:", len(daily))